# 特徴抽出パイプライン

LLM特徴量 + 言語特徴量を抽出して `data/processed/features.csv` に保存する。

In [ ]:
import sys; sys.path.insert(0, '..')
from transformers import pipeline as hf_pipeline, BitsAndBytesConfig
import torch, pandas as pd
from src.preprocessing.data_loader import load_dialogues
from src.preprocessing.text_normalizer import normalize
from src.preprocessing.tokenizer import pos_tag
from src.features.llm_features import extract_llm_features
from src.features.linguistic_features import extract_linguistic_features
from src.features.feature_merger import merge_features

In [ ]:
# モデル名は config で切り替え
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
pipe = hf_pipeline('text-generation', model=MODEL_NAME, model_kwargs={'quantization_config': bnb_config}, device_map='auto')

In [ ]:
dialogues = load_dialogues()
llm_rows, ling_rows, labels = [], [], []
for d in dialogues:
    text = normalize(d.text)
    tagged = pos_tag(text)
    llm_feat = extract_llm_features(text, pipe)
    ling_feat = extract_linguistic_features(text, tagged)
    llm_rows.append(llm_feat.to_dict())
    ling_rows.append(ling_feat.to_dict())
    labels.append(d.label)

df = merge_features(llm_rows, ling_rows)
df['label'] = labels
df.to_csv('../data/processed/features.csv', index=False)
print(df.shape)